In [1]:
from google.colab import files
files.upload()

Output hidden; open in https://colab.research.google.com to view.

In [3]:
import pandas as pd
df = pd.read_excel("Indian_Defence_Stocks_Ultimate_LSTM_Dataset_FINAL.xlsx.xlsx")

print(df.shape)
df.head()

(27810, 59)


,Date,Close,High,Low,Open,Volume,Return,MA_5,MA_20,MA_50,...,Volume_MA20,Volume_Growth,Dist_52W_High,Dist_52W_Low,WeekOfYear,MonthEnd,QuarterEnd,COVID_2020,GFC_2008,Target
0,2019-07-15,134.172287,137.391699,133.832209,135.124509,28798,-0.014488,135.954300,142.242346,136.411357,...,246097.7,-0.390596,-0.198550,0.338509,29,0,0,0,0,0
1,2019-07-16,133.696152,136.824870,132.970648,134.172256,49192,-0.003549,135.124500,141.850122,136.461235,...,212148.6,0.708174,-0.201394,0.333759,29,0,0,0,0,1
2,2019-07-17,137.595749,140.021631,134.671069,134.671069,88530,0.029168,135.491791,141.648343,136.659388,...,200451.3,0.799683,-0.178101,0.372662,29,0,0,0,0,0
3,2019-07-18,136.734192,139.658872,136.031369,137.595735,43304,-0.006262,135.668628,141.449963,136.918755,...,198785.7,-0.510855,-0.183247,0.364067,29,0,0,0,0,1
4,2019-07-19,138.162552,142.787611,135.124512,136.734211,117282,0.010446,136.072186,141.431827,137.223919,...,201796.0,1.708341,-0.174715,0.378316,29,0,0,0,0,0


In [4]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27810 entries, 0 to 27809
Data columns (total 59 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Date                27810 non-null  datetime64[ns]
 1   Close               27810 non-null  float64       
 2   High                27810 non-null  float64       
 3   Low                 27810 non-null  float64       
 4   Open                27810 non-null  float64       
 5   Volume              27810 non-null  int64         
 6   Return              27810 non-null  float64       
 7   MA_5                27810 non-null  float64       
 8   MA_20               27810 non-null  float64       
 9   MA_50               27810 non-null  float64       
 10  Volatility_5        27810 non-null  float64       
 11  Volatility_20       27810 non-null  float64       
 12  HL_PCT              27810 non-null  float64       
 13  CO_PCT              27810 non-null  float64   

In [5]:
print(df.isnull().sum())

Date                  0
Close                 0
High                  0
Low                   0
Open                  0
Volume                0
Return                0
MA_5                  0
MA_20                 0
MA_50                 0
Volatility_5          0
Volatility_20         0
HL_PCT                0
CO_PCT                0
Stock                 0
Group                 0
EMA_12                0
EMA_26                0
MACD                  0
Signal_Line           0
BB_Middle             0
BB_Upper              0
BB_Lower              0
RSI                   0
Momentum_10           0
Price_Change_PCT      0
Daily_Range           0
Gap_PCT               0
Rel_Volume            0
Sharpe_20             0
DayOfWeek             0
Month                 0
Quarter               0
Year                  0
NIFTY_Close           0
NIFTY_Return          0
NIFTY_MA20            0
NIFTY_Volatility20    0
RS_vs_NIFTY           0
Close_Lag_1           0
Return_Lag_1          0
Close_Lag_2     

In [6]:
print(df['Target'].value_counts())

Target
0    14155
1    13655
Name: count, dtype: int64


In [7]:
print(df['Target'].value_counts())

Target
0    14155
1    13655
Name: count, dtype: int64


In [8]:
print(df['Target'].value_counts(normalize=True))

Target
0    0.50899
1    0.49101
Name: proportion, dtype: float64


In [9]:
df = df.sort_values("Date").reset_index(drop=True)

X = df.drop(columns=["Target", "Date"])
y = df["Target"]

split = int(len(df) * 0.8)

X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

In [13]:
cols_to_drop = [
    "Close", "Open", "High", "Low"
]

In [14]:
df = df.sort_values(["Stock", "Date"]).reset_index(drop=True)

In [15]:
df = df.drop(columns=[
    "Date",          # time leakage risk in ML models
    "Group"          # can create hidden bias
])

In [16]:
df = df.drop(columns=[
    "Open", "High", "Low", "Close"
])

In [17]:
df = df.drop(columns=[
    "Close_Lag_2", "Close_Lag_3", "Close_Lag_10",
    "Return_Lag_2", "Return_Lag_3", "Return_Lag_10"
])

In [18]:
df = df.drop(columns=[
    "NIFTY_MA20"
])

In [19]:
num_cols = df.select_dtypes(include=["float64", "int64"]).columns

for col in num_cols:
    lower = df[col].quantile(0.01)
    upper = df[col].quantile(0.99)
    df[col] = df[col].clip(lower, upper)

In [20]:
import numpy as np

corr = df.corr(numeric_only=True)

upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

to_drop = [
    col for col in upper.columns
    if any(abs(upper[col]) > 0.95)
]

df = df.drop(columns=to_drop)

In [21]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df["Stock"] = le.fit_transform(df["Stock"])

In [22]:
df = df.replace([np.inf, -np.inf], np.nan)
df = df.ffill().bfill()
df = df.dropna()

In [23]:
smooth_cols = ["Volume", "Rel_Volume", "Volatility_5", "Volatility_20"]

for col in smooth_cols:
    df[col] = df[col].rolling(window=3, min_periods=1).mean()

In [24]:
print(df.shape)
print(df.isnull().sum().sum())
print(df.describe())

(27810, 33)
0
             Volume        Return          MA_5  Volatility_5  Volatility_20  \
count  2.781000e+04  27810.000000  27810.000000  27810.000000   27810.000000   
mean   4.014666e+06      0.001070    733.032821      0.021377       0.023459   
std    7.193590e+06      0.023617    907.729382      0.012664       0.010540   
min    3.379434e+04     -0.061581      8.371863      0.003978       0.008266   
25%    6.314865e+05     -0.012136    152.601189      0.012579       0.015866   
50%    1.552244e+06     -0.000342    405.548251      0.018145       0.021141   
75%    3.382481e+06      0.012544    893.572232      0.026508       0.028336   
max    4.663004e+07      0.082550   4383.886443      0.076161       0.062261   

             HL_PCT        CO_PCT         Stock          MACD           RSI  \
count  27810.000000  27810.000000  27810.000000  27810.000000  27810.000000   
mean       0.035980     -0.001370      3.899784      4.337425     51.781997   
std        0.021638      0.0

In [25]:
import numpy as np

for col in ["Volume", "Volume_MA20"]:
    df[col] = np.log1p(df[col])

In [26]:
ratio_cols = [
    "Rel_Volume",
    "Volume_Growth",
    "Sharpe_20"
]

df[ratio_cols] = df[ratio_cols].replace([np.inf, -np.inf], np.nan)
df[ratio_cols] = df[ratio_cols].fillna(0)

In [27]:
df[ratio_cols] = df[ratio_cols].clip(-3, 3)

In [28]:
df["MACD"] = df["MACD"] / (df["MACD"].std() + 1e-6)

In [29]:
df["RSI"] = df["RSI"] / 100.0

In [30]:
low_variance_cols = [
    col for col in df.columns
    if df[col].std() < 0.01 and col not in ["Target"]
]

df = df.drop(columns=low_variance_cols)

In [31]:
corr = df.corr(numeric_only=True)

upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

to_drop = [
    col for col in upper.columns
    if any(abs(upper[col]) > 0.92)
]

df = df.drop(columns=to_drop)

In [32]:
df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna()
df = df.reset_index(drop=True)

In [34]:
X = df.drop(columns=["Target"])
y = df["Target"]

split = int(len(df) * 0.8)

X_train = X.iloc[:split]
X_test  = X.iloc[split:]
y_train = y.iloc[:split]
y_test  = y.iloc[split:]

In [35]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=600,
    max_depth=6,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42
)

xgb.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.03, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=600, n_jobs=None,
              num_parallel_tree=None, ...)

In [37]:
!pip install catboost
from catboost import CatBoostClassifier

cat = CatBoostClassifier(
    iterations=800,
    depth=6,
    learning_rate=0.03,
    loss_function="Logloss",
    verbose=0
)

cat.fit(X_train, y_train)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.5 MB/s eta 0:00:00


CatBoostClassifier(depth=6, iterations=800, learning_rate=0.03, loss_function='Logloss', verbose=0)

In [38]:
import numpy as np

xgb_pred = xgb.predict_proba(X_test)[:, 1]
cat_pred = cat.predict_proba(X_test)[:, 1]

final_pred = (xgb_pred + cat_pred) / 2
final_pred = (final_pred > 0.5).astype(int)

In [39]:
from sklearn.metrics import accuracy_score, classification_report

print("Accuracy:", accuracy_score(y_test, final_pred))
print(classification_report(y_test, final_pred))

Accuracy: 0.5816253146350234
              precision    recall  f1-score   support

           0       0.59      0.59      0.59      2814
           1       0.58      0.57      0.58      2748

    accuracy                           0.58      5562
   macro avg       0.58      0.58      0.58      5562
weighted avg       0.58      0.58      0.58      5562



In [41]:
print(df.shape)
print(df.columns)
print(df["Target"].value_counts(normalize=True))
print(df.isnull().sum().sum())

(27810, 29)
Index(['Volume', 'Return', 'MA_5', 'Volatility_5', 'Volatility_20', 'HL_PCT',
       'CO_PCT', 'Stock', 'MACD', 'RSI', 'Momentum_10', 'Daily_Range',
       'Rel_Volume', 'Sharpe_20', 'DayOfWeek', 'Month', 'Year', 'NIFTY_Return',
       'RS_vs_NIFTY', 'Return_Lag_1', 'Return_Lag_5', 'Volume_Growth',
       'Dist_52W_High', 'Dist_52W_Low', 'MonthEnd', 'QuarterEnd', 'COVID_2020',
       'GFC_2008', 'Target'],
      dtype='object')
Target
0    0.50899
1    0.49101
Name: proportion, dtype: float64
0


In [42]:
features = [
    # momentum / trend
    "Return",
    "RSI",
    "MACD",
    "Signal_Line",

    # moving averages
    "MA_5",
    "MA_20",
    "EMA_12",
    "EMA_26",

    # volatility
    "Volatility_5",
    "Volatility_20",
    "HL_PCT",

    # lag signals
    "Close_Lag_1",
    "Return_Lag_1",
    "Close_Lag_5",
    "Return_Lag_5",

    # market context
    "NIFTY_Return",
    "RS_vs_NIFTY",

    # volume
    "Rel_Volume",
    "Volume_Growth",

    # stock identity
    "Stock"
]

In [44]:
existing_cols = df.columns.tolist()
filtered_features = [col for col in features if col in existing_cols]
X = df[filtered_features]
y = df["Target"]

In [45]:
split = int(len(df) * 0.8)

X_train = X.iloc[:split]
X_test  = X.iloc[split:]
y_train = y.iloc[:split]
y_test  = y.iloc[split:]

In [46]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=500,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42
)

model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=5, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=500, n_jobs=None,
              num_parallel_tree=None, ...)

In [47]:
from sklearn.metrics import accuracy_score, classification_report

pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred))

Accuracy: 0.5510607695073715
              precision    recall  f1-score   support

           0       0.55      0.63      0.59      2814
           1       0.55      0.47      0.51      2748

    accuracy                           0.55      5562
   macro avg       0.55      0.55      0.55      5562
weighted avg       0.55      0.55      0.55      5562



In [48]:
features = [
    # CORE PRICE ACTION
    "Return",
    "Close_Lag_1",
    "Close_Lag_2",
    "Close_Lag_5",

    # TREND
    "MA_5",
    "MA_20",
    "EMA_12",
    "EMA_26",

    # MOMENTUM
    "RSI",
    "MACD",
    "Momentum_10",

    # VOLATILITY
    "Volatility_5",
    "Volatility_20",
    "HL_PCT",
    "CO_PCT",

    # VOLUME SIGNALS
    "Volume",
    "Rel_Volume",
    "Volume_Growth",

    # MARKET CONTEXT
    "NIFTY_Return",
    "RS_vs_NIFTY",

    # IDENTITY
    "Stock"
]

In [49]:
print(df[["Return", "Target"]].head(10))
print(df[["Close_Lag_1"]].head(10))

     Return  Target
0 -0.014488       0
1 -0.003549       1
2  0.029168       0
3 -0.006262       1
4  0.010446       0
5 -0.015589       1
6  0.006334       0
7 -0.011595       0
8 -0.013072       1
9  0.020717       1


KeyError: "None of [Index(['Close_Lag_1'], dtype='object')] are in the [columns]"

In [50]:
df = df.sort_values("Date").reset_index(drop=True)

df["Close_Lag_1"] = df["Close"].shift(1)
df["Close_Lag_5"] = df["Close"].shift(5)

df["Return_Lag_1"] = df["Return"].shift(1)
df["Return_Lag_5"] = df["Return"].shift(5)

KeyError: 'Date'

In [51]:
print(df.columns)

Index(['Volume', 'Return', 'MA_5', 'Volatility_5', 'Volatility_20', 'HL_PCT',
       'CO_PCT', 'Stock', 'MACD', 'RSI', 'Momentum_10', 'Daily_Range',
       'Rel_Volume', 'Sharpe_20', 'DayOfWeek', 'Month', 'Year', 'NIFTY_Return',
       'RS_vs_NIFTY', 'Return_Lag_1', 'Return_Lag_5', 'Volume_Growth',
       'Dist_52W_High', 'Dist_52W_Low', 'MonthEnd', 'QuarterEnd', 'COVID_2020',
       'GFC_2008', 'Target'],
      dtype='object')


In [52]:
print(df.head())
print(df.index)

      Volume    Return        MA_5  Volatility_5  Volatility_20    HL_PCT  \
0  10.428078 -0.014488  135.954300      0.014751       0.030280  0.026529   
1  10.633308 -0.003549  135.124500      0.011503       0.028701  0.028828   
2  10.953839  0.029168  135.491791      0.013042       0.028477  0.038886   
3  11.007800 -0.006262  135.668628      0.013671       0.027727  0.026530   
4  11.327074  0.010446  136.072186      0.016631       0.027994  0.055464   

     CO_PCT  Stock      MACD       RSI  ...  Return_Lag_1  Return_Lag_5  \
0 -0.007047      0 -0.009460  0.434735  ...      0.002170     -0.061457   
1 -0.003548      0 -0.024911  0.416575  ...     -0.014488      0.020820   
2  0.021717      0 -0.025938  0.441566  ...     -0.003549     -0.015132   
3 -0.006261      0 -0.028832  0.231743  ...      0.029168      0.000668   
4  0.010446      0 -0.026817  0.268078  ...     -0.006262      0.002170   

   Volume_Growth  Dist_52W_High  Dist_52W_Low  MonthEnd  QuarterEnd  \
0      -0.39059

In [56]:
df = pd.read_excel("Indian_Defence_Stocks_Ultimate_LSTM_Dataset_FINAL.xlsx.xlsx")

In [55]:
df = df.sort_values("Date").reset_index(drop=True)

In [58]:
df["Close_Lag_1"] = df["Close"].shift(1)
df["Close_Lag_5"] = df["Close"].shift(5)

df["Return_Lag_1"] = df["Return"].shift(1)
df["Return_Lag_5"] = df["Return"].shift(5)

df = df.dropna().reset_index(drop=True)

In [59]:
print(df.columns)

Index(['Date', 'Close', 'High', 'Low', 'Open', 'Volume', 'Return', 'MA_5',
       'MA_20', 'MA_50', 'Volatility_5', 'Volatility_20', 'HL_PCT', 'CO_PCT',
       'Stock', 'Group', 'EMA_12', 'EMA_26', 'MACD', 'Signal_Line',
       'BB_Middle', 'BB_Upper', 'BB_Lower', 'RSI', 'Momentum_10',
       'Price_Change_PCT', 'Daily_Range', 'Gap_PCT', 'Rel_Volume', 'Sharpe_20',
       'DayOfWeek', 'Month', 'Quarter', 'Year', 'NIFTY_Close', 'NIFTY_Return',
       'NIFTY_MA20', 'NIFTY_Volatility20', 'RS_vs_NIFTY', 'Close_Lag_1',
       'Return_Lag_1', 'Close_Lag_2', 'Return_Lag_2', 'Close_Lag_3',
       'Return_Lag_3', 'Close_Lag_5', 'Return_Lag_5', 'Close_Lag_10',
       'Return_Lag_10', 'Volume_MA20', 'Volume_Growth', 'Dist_52W_High',
       'Dist_52W_Low', 'WeekOfYear', 'MonthEnd', 'QuarterEnd', 'COVID_2020',
       'GFC_2008', 'Target'],
      dtype='object')


In [60]:
df = df.sort_values("Date").reset_index(drop=True)

In [61]:
features = [
    # PRICE ACTION
    "Return",
    "Close_Lag_1",
    "Close_Lag_5",
    "Return_Lag_1",
    "Return_Lag_5",

    # TREND
    "MA_5",
    "MA_20",
    "EMA_12",
    "EMA_26",

    # MOMENTUM
    "RSI",
    "MACD",
    "Signal_Line",
    "Momentum_10",

    # VOLATILITY
    "Volatility_5",
    "Volatility_20",
    "HL_PCT",
    "CO_PCT",

    # VOLUME
    "Volume",
    "Rel_Volume",
    "Volume_Growth",
    "Volume_MA20",

    # MARKET
    "NIFTY_Return",
    "RS_vs_NIFTY",

    # TIME (SAFE ONES ONLY)
    "DayOfWeek",
    "WeekOfYear",

    # IDENTITY
    "Stock",

    # REGIME
    "COVID_2020",
    "GFC_2008"
]

In [62]:
X = df[features]
y = df["Target"]

split = int(len(df) * 0.8)

X_train = X.iloc[:split]
X_test = X.iloc[split:]
y_train = y.iloc[:split]
y_test = y.iloc[split:]

In [63]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=800,
    max_depth=6,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42
)

model.fit(X_train, y_train)

ValueError: DataFrame.dtypes for data must be int, float, bool or category. When categorical type is supplied, the experimental DMatrix parameter`enable_categorical` must be set to `True`.  Invalid columns:Stock: object

In [64]:
X_train = X_train.drop(columns=["Stock"])
X_test = X_test.drop(columns=["Stock"])

In [65]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["Stock"] = le.fit_transform(df["Stock"])

In [66]:
model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.03, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=800, n_jobs=None,
              num_parallel_tree=None, ...)

In [67]:
from sklearn.metrics import accuracy_score

pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred))

Accuracy: 0.5140287769784173
